# Task 03 — Decision Tree Classifier
### SkillCraft Technology Internship

**Objective:** Predict whether a customer subscribes to a bank term deposit using the UCI Bank Marketing Dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

IMG_DIR = 'images'
os.makedirs(IMG_DIR, exist_ok=True)
print('Libraries loaded.')

## Step 1 — Load Dataset

In [ ]:
df = pd.read_csv('bank-additional-full.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()

## Step 2 — EDA

In [ ]:
print(df['y'].value_counts())
fig, ax = plt.subplots(figsize=(6,4))
counts = df['y'].value_counts()
ax.bar(counts.index, counts.values, color=['#5C85D6','#EF5350'], edgecolor='white', width=0.5)
ax.set_title('Target Class Distribution', fontweight='bold')
ax.set_xlabel('Subscribed')
ax.set_ylabel('Count')
ax.set_xticks([0,1]); ax.set_xticklabels(['No','Yes'])
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(df['age'], bins=40, color='#5C85D6', edgecolor='white', alpha=0.85)
ax.set_title('Age Distribution', fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Preprocessing

In [ ]:
label_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
df_encoded = df.copy()
for col in label_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])
df_encoded.to_csv('bank_cleaned.csv', index=False)
print('Saved bank_cleaned.csv')
df_encoded.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14,10))
corr = df_encoded.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Train / Test Split

In [ ]:
X = df_encoded.drop(columns=['y'])
y = df_encoded['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

## Step 5 — Optimal Depth

In [ ]:
depths = range(3,11)
train_s, test_s = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=42)
    m.fit(X_train, y_train)
    train_s.append(accuracy_score(y_train, m.predict(X_train)))
    test_s.append(accuracy_score(y_test, m.predict(X_test)))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(depths, [s*100 for s in train_s], 'o-', label='Train', color='#5C85D6')
ax.plot(depths, [s*100 for s in test_s], 's--', label='Test', color='#EF5350')
ax.axvline(x=5, color='gray', linestyle=':', label='depth=5')
ax.set_title('Accuracy vs Tree Depth', fontweight='bold')
ax.set_xlabel('Max Depth'); ax.set_ylabel('Accuracy (%)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/depth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Train Model & Evaluate

In [ ]:
clf = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f'Train Acc: {accuracy_score(y_train,clf.predict(X_train))*100:.2f}%')
print(f'Test Acc : {accuracy_score(y_test,y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=['No','Yes']))

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No','Yes'], yticklabels=['No','Yes'],
            linewidths=0.5, linecolor='white', annot_kws={'size':14})
ax.set_title('Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
y_prob = clf.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, color='#5C85D6', lw=2, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1],'--', color='gray')
ax.set_title('ROC Curve', fontweight='bold')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC: {roc_auc:.4f}')

In [ ]:
fi = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,5))
fi.head(10).plot(kind='bar', ax=ax, color='#5C85D6', edgecolor='white')
ax.set_title('Top 10 Feature Importances', fontweight='bold')
ax.set_xlabel('Feature'); ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8,6))
fi.sort_values(ascending=True).tail(10).plot(kind='barh', ax=ax, color='#5C85D6', edgecolor='white')
ax.set_title('Top 10 Feature Importances (Horizontal)', fontweight='bold')
ax.set_xlabel('Score')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/feature_importances_horizontal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(22,10))
plot_tree(clf, feature_names=X.columns.tolist(), class_names=['No','Yes'],
          filled=True, rounded=True, fontsize=7, ax=ax, max_depth=3)
plt.title('Decision Tree Visualization (Depth 3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()